In [ ]:
# LabOne Q:
from laboneq.simple import *

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pathlib import Path
import time
import os
import pandas as pd
import laboneq
from datetime import datetime
from laboneq.contrib.example_helpers.plotting.plot_helpers import (
    plot_results,
    plot_simulation,
)
from laboneq.analysis.fitting import oscillatory

print(laboneq.__version__)

In [ ]:
"""Descriptor for a QCCS consisting of a single SHFQC
"""
descriptor_shfqc = """ 
instruments:
  SHFQC:
  - address: DEV12296
    uid: device_shfqc

connections:
  device_shfqc:
    - iq_signal: q0/drive_line
      ports: SGCHANNELS/1/OUTPUT
    - iq_signal: q0/measure_line
      ports: [QACHANNELS/0/OUTPUT]
    - acquire_signal: q0/acquire_line
      ports: [QACHANNELS/0/INPUT]
"""
# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    descriptor_shfqc,
    server_host="127.0.0.1",  # ip address of the LabOne dataserver used to communicate with the instruments
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="UCLA_SHFQC",  # setup name
)
# Are we emulating? or actually creating pulses?
emulate = False
# create and connect to session
session = Session(device_setup=device_setup)
session.connect(do_emulation=emulate)

In [ ]:
res_lo_freq = 7.2e9
qb_lo_freq = 3e9
qb_freq = 3.1027e9
res_freq = 7.14882e9
integration_time = 1e-5

time_start = 0
time_stop = 0.5e-6
length_num = 201

ro_pulse_amp = 0.1
ro_pulse_length = 5e-6
drive_pulse_amp = 1
drive_pulse_length = 5e-6
drive_range = 0
num_averages = 10  # average number = 2^n, n_max = 17
measure_range = -20  # actual power = range (dBm) - 20log(amp)
measure_amp = 0.9
acquire_range = -40
acquire_amp = 1
integration_time = 1e-5


In [ ]:
# range of pulse amplitude scan
def create_rabi_length_sweep(time_start, time_stop, length_num, uid="rabi_length"):
    return LinearSweepParameter(uid="length", start=time_start, stop=time_stop, count=length_num)


def create_readout_pulse(
    qubit, length=ro_pulse_length, amplitude=ro_pulse_amp):
    readout_pulse = pulse_library.const(
        uid=f"readout_pulse_{qubit}",
        length=length,
        amplitude=amplitude)
    return readout_pulse


def create_rabi_drive_pulse(qubit, length=drive_pulse_length, amplitude=drive_pulse_amp):
    return pulse_library.const(
        uid=f"gaussian_drive_q{qubit}", length=length, amplitude=amplitude
    )

In [ ]:

def length_rabi(drive_pulse, readout_pulse, length_sweep):
    exp_rabi = Experiment(
        uid="Amplitude Rabi",
        signals=[
            ExperimentSignal("drive"),
            ExperimentSignal("measure"),
            ExperimentSignal("acquire"),
        ],
    )

    ## define Rabi experiment pulse sequence
    # outer loop - real-time, cyclic averaging
    with exp_rabi.acquire_loop_rt(
        uid="rabi_shots",
        count=pow(2, num_averages),
        acquisition_type=AcquisitionType.SPECTROSCOPY,
    ):
        # inner loop - real time sweep of Rabi ampitudes
        with exp_rabi.sweep(uid="rabi_sweep", parameter=length_sweep):
            # play qubit excitation pulse - pulse amplitude is swept
            with exp_rabi.section(
                uid="qubit_excitation"):
                exp_rabi.play(
                    signal="drive", pulse=drive_pulse, length = length_sweep
                )
            # readout pulse and data acquisition
            with exp_rabi.section(uid="readout_section", play_after="qubit_excitation"):
                # play readout pulse on measure line
                exp_rabi.play(signal="measure", pulse=readout_pulse)
                # trigger signal data acquisition
                exp_rabi.measure(
                    acquire_signal="acquire",
                    integration_length = integration_time,
                    handle="length_rabi",
                    reset_delay = 50e-6,
                )
            # relax time after readout - for qubit relaxation to groundstate and signal processing
            with exp_rabi.section(uid="reserve", length=150e-6):
                exp_rabi.reserve(signal="measure")
    return exp_rabi

In [ ]:
# define pulses and create experiment
readout_pulse = create_readout_pulse("q0")
drive_pulse = create_rabi_drive_pulse("q0")
exp_rabi = length_rabi(drive_pulse, readout_pulse, create_rabi_length_sweep(time_start, time_stop, length_num))


# signal map for qubit 0
def signal_map_default(qubit):
    signal_map = {
        "drive": device_setup.logical_signal_groups[f"q0"].logical_signals[
            "drive_line"
        ],
        "measure": device_setup.logical_signal_groups[f"q0"].logical_signals[
            "measure_line"
        ],
        "acquire": device_setup.logical_signal_groups[f"q0"].logical_signals[
            "acquire_line"
        ],
    }
    return signal_map


# run the experiment on qubit 0
exp_rabi.set_signal_map(signal_map_default("q0"))

In [ ]:
exp_calibration = Calibration()
exp_calibration["drive"] = SignalCalibration(
    oscillator = Oscillator(uid = "ch0_osc_0", frequency = qb_freq, 
        modulation_type=ModulationType.HARDWARE
    ),
    local_oscillator = Oscillator(uid="ch0_lo", frequency = qb_lo_freq),
    range = drive_range,
)
exp_calibration["measure"] = SignalCalibration(
    oscillator = Oscillator(uid = "qa_osc_0", frequency = res_freq,
        modulation_type=ModulationType.HARDWARE
    ),
    local_oscillator = Oscillator(uid="qa_lo", frequency = res_lo_freq),
    range = measure_range
)
exp_calibration["acquire"] = SignalCalibration(
    range = acquire_range
)
exp_rabi.set_calibration(exp_calibration)

In [ ]:
# compile the experiment on the open instrument session
compiled_rabi = session.compile(exp_rabi)

Path("Pulse_Sheets").mkdir(parents=True, exist_ok=True)
# generate a pulse sheet to inspect experiment before runtime
show_pulse_sheet("Pulse_Sheets/Rabi_length", compiled_rabi)

In [ ]:
plot_simulation(compiled_rabi, start_time=600e-6, length=25e-6)

In [ ]:
# run the compiled experiemnt
rabi_results = session.run()
timestamp = time.strftime("%Y%m%dT%H%M%S")

# --- Make output folder + timestamp ---
outdir = "rabi_results"
os.makedirs(outdir, exist_ok=True)


In [ ]:
# --- Convert to dBm for the trace (keep your original definition) ---
def to_dbm(signal):
    power_W = (signal**2) / 50.0
    return 10 * np.log10(np.maximum(power_W, 1e-20) / 1e-3)  # avoid log(0)

# --- Data acquisition ---
rabi_res = rabi_results.get_data("amp_rabi")   # complex data
rabi_amp = rabi_results.get_axis("amp_rabi")[0]

# --- Compute quantities ---
amp_abs   = np.abs(rabi_res)
amp_dbm   = to_dbm(amp_abs)
phase_rad = np.angle(rabi_res)
real_dbm  = to_dbm(np.abs(rabi_res.real))
imag_dbm  = to_dbm(np.abs(rabi_res.imag))

# ==========================
# Sophisticated fit function
# ==========================
def rabi_power_model(A, c0, c1, c2, B, A_pi, A0, Adec, p):
    baseline = c0 + c1*A + c2*(A**2)
    osc = B * np.cos(2*np.pi*(A - A0)/np.maximum(A_pi, 1e-12)) \
            * np.exp(- (np.maximum(A,0)/np.maximum(Adec,1e-12))**p)
    return baseline + osc

# --- Initial guesses ---
A = np.asarray(rabi_amp).astype(float)
Y = np.asarray(amp_dbm).astype(float)

coefs = np.polyfit(A, Y, deg=2)
c2_0, c1_0, c0_0 = coefs
B_0 = 0.5 * (np.nanmax(Y) - np.nanmin(Y))

# estimate period
detr = Y - (c0_0 + c1_0*A + c2_0*A**2)
pk, _ = find_peaks(detr, distance=max(2, len(A)//20))
if len(pk) >= 2:
    A_pi_0 = np.median(np.diff(A[pk]))
else:
    A_pi_0 = (A.max() - A.min()) / 4.0

A0_0   = A.min() + 0.25*(A.max()-A.min())
Adec_0 = 0.5*(A.max()-A.min())
p_0    = 1.5

p0 = [c0_0, c1_0, c2_0, B_0, A_pi_0, A0_0, Adec_0, p_0]
lower = [-np.inf, -np.inf, -np.inf, -np.inf, 1e-6, A.min()-2*(A.max()-A.min()), 1e-6, 0.5]
upper = [ np.inf,  np.inf,  np.inf,  np.inf, (A.max()-A.min())*5, A.max()+2*(A.max()-A.min()), (A.max()-A.min())*10, 3.0]

# --- Fit ---
popt, pcov = curve_fit(rabi_power_model, A, Y, p0=p0, bounds=(lower, upper), maxfev=20000)
c0, c1, c2, B, A_pi_fit, A0_fit, Adec_fit, p_fit = popt

# --- Compute fitted curve ---
A_plot = np.linspace(A.min(), A.max(), 5*len(A))
Y_fit  = rabi_power_model(A_plot, *popt)

# --- Find π pulse as the FIRST MAXIMUM of the fitted curve in-range ---
pk, _ = find_peaks(Y_fit, distance=max(2, len(A_plot)//20))
if len(pk) > 0:
    idx_pi = int(pk[0])               # first peak along the sweep
else:
    idx_pi = int(np.argmax(Y_fit))    # fallback

A_pi_point = float(A_plot[idx_pi])    # amplitude (a.u.)
dbm_pi     = 20*np.log10(max(A_pi_point, 1e-12))  # keep your convention

# ===================
# PLOTTING
# ===================
fig, axs = plt.subplots(2, 1, figsize=(6, 4.5),
                        sharex=True, gridspec_kw={'height_ratios': [1.5, 1.0]})

# Amplitude (dBm) + fit
axs[0].plot(rabi_amp, amp_dbm, 'o', ms=3, label="data")
axs[0].plot(A_plot, Y_fit, '-r', lw=1.8, label="fit")
axs[0].axvline(A_pi_point, color="grey", linestyle="--", linewidth=1.5)
axs[0].set_ylabel("Power (dBm)")
axs[0].legend([rf"$\pi$ pulse: $A_\pi$={A_pi_point:.3f} → {dbm_pi:.1f} dBm"],
              loc="lower right", framealpha=0.9)
axs[0].set_title("Length Rabi Q1", fontsize=12, pad=8)

# Phase
axs[1].plot(rabi_amp, phase_rad, 'o', ms=3, color="tab:purple")
axs[1].set_xlabel("Drive amplitude (a.u.)")
axs[1].set_ylabel("Phase (rad)")

plt.tight_layout()

# --- Save figure ---
fig_path = os.path.join(outdir, f"rabi_experiment_length_{timestamp}Q1.png")
plt.savefig(fig_path, dpi=600)
print(f"Saved figure to {fig_path}")

# --- Save data to CSV ---
df = pd.DataFrame({
    "DriveAmplitude": rabi_amp,
    "Amplitude_dBm": amp_dbm,
    "Real_dBm": real_dbm,
    "Imag_dBm": imag_dbm,
    "Phase_rad": phase_rad,
    "Fit_y(A)": rabi_power_model(rabi_amp, *popt),
})
csv_path = os.path.join(outdir, f"rabi_experiment_length_{timestamp}Q1.csv")
df.to_csv(csv_path, index=False)
print(f"Saved data to {csv_path}")

plt.show()